# Step 1: Load and Clean Data

Load raw NASDAQ data, clean and validate, handle missing values.

## Steps
1. Load all data (ETFs, stocks, metadata, companies) from CSV files into dictionary format
2. Clean at dictionary level: remove duplicates, numeric conversion, and sort by date
3. Repair OHLCV data quality 
4. Combine cleaned tickers into unified DataFrame
5. Apply final unified cleaning (cross-ticker duplicate checks)
6. Fill missing data using local interpolation (5-day window) and drop tickers with >10% missing data
7. Validate data quality through comprehensive validation report
8. Save cleaned data to `data/processed/nasdaq_processed.csv` 


In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path('..').resolve()))

from src.data.loaders.loader import load_all_data, combine_dataframes_to_unified
from src.data.cleaning.clean import (
    clean_ticker_data,
    clean_unified_dataframe,
    fill_missing_data_unified,
)
from src.data.cleaning.repair import repair_ohlcv_quality
from src.data.cleaning.validation import generate_validation_report


## 1. Load Data


In [2]:
# Load all data (uses default path: data/raw/stock-market-dataset)
etfs_dict, stocks_dict, symbols_meta, companies = load_all_data()


Loading stock data: 100%|██████████| 5884/5884 [00:17<00:00, 330.66it/s]


## 2. Clean Data

Clean each ticker's data individually before combining into unified DataFrame.


In [ ]:
# Basic cleaning (duplicates, numeric conversion, sorting)
stocks_dict = clean_ticker_data(stocks_dict, label='stock')


Combining tables: 100%|██████████| 4339/4339 [00:00<00:00, 4605031.64it/s]


## 3. Repair OHLCV data quality

In [ ]:
# Repairing OHLCV quality 
repair_ohlcv_quality(stocks_dict, label='stock')


## 4. Combine cleaned tickers into unified DataFrame

In [ ]:
# Combining cleaned tickers into unified DataFrame
unified_df = combine_dataframes_to_unified(stocks_dict)


## 5. Final unified cleaning 
Cleaning and missing data handling on the unified DataFrame (cross-ticker duplicate checks).

In [4]:
# Final unified DataFrame cleaning (cross-ticker duplicate check)
cleaned_df = clean_unified_dataframe(unified_df)


# Missing data handling strategy:
# 1. Check for missing data (skip if none)
# 2. Interpolate gaps using nearby values (5-day window)
# 3. Drop tickers with >10% missing data remaining
# Missing values are handled using a 5-day local interpolation window.

# Filling missing data (interpolation-only, fast)
cleaned_df = fill_missing_data_unified(cleaned_df)


Dropping 175 tickers with >10% missing data
Removed 4,399 rows with negative prices across 6 tickers


## 6. Validate Cleaned Data


In [5]:
# Generate validation report
validation_report = generate_validation_report(cleaned_df, verbose=True)



Data validation report
Total rows: 12,993,961
Total tickers: 4,164
Date range: 1962-01-02 to 2020-04-01
Unique dates: 14,663

Missing values:
  Volume: 32,096 (0.25%)

Zero prices in Adj Close: 0
Negative prices in Adj Close: 0
Invalid dates: 0

Adj Close statistics:
  Mean: $31.42
  Median: $11.43
  Min: $0.00
  Max: $38153.69

  Found 4,655 rows with prices >$10,000
  Examples: CPST, ELC, KTB, NTB, SAVA

  Found 29,152 rows with prices <$0.01 (penny stocks)
  Examples: AQB, AWR, BTI, CCD, EAF

First few rows:
  ticker       Date       Open       High        Low      Close  Adj Close  \
0      A 1999-11-18  32.546494  35.765381  28.612303  31.473534  27.068665   
1      A 1999-11-19  30.713520  30.758226  28.478184  28.880543  24.838577   
2      A 1999-11-22  29.551144  31.473534  28.657009  31.473534  27.068665   
3      A 1999-11-23  30.400572  31.205294  28.612303  28.612303  24.607880   
4      A 1999-11-24  28.701717  29.998211  28.612303  29.372318  25.261524   

       Volume

## 7. Save Cleaned Data
A portion of the data is saved in a separate file to be used on the website.

In [6]:
output_path = Path('../data/processed/nasdaq_processed.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

overwrite = True

if output_path.exists() and not overwrite:
    existing_size = output_path.stat().st_size / 1e9
    print(f"File already exists ({existing_size:.2f} GB)")
    print("Set overwrite=True to save new version")
else:
    if output_path.exists():
        existing_size = output_path.stat().st_size / 1e9
        print(f"Overwriting existing file ({existing_size:.2f} GB)")
    
    print("Saving cleaned data...")
    cleaned_df.to_csv(output_path, index=False)
    file_size_gb = output_path.stat().st_size / 1e9
    print(f"Saved {len(cleaned_df):,} rows to {output_path}")
    print(f"File size: {file_size_gb:.2f} GB")

print(f"\nColumns saved: {', '.join(cleaned_df.columns.tolist())}")
print(f"\nCleaned data saved to {output_path}")

# Define path filtered datasets
lite_output_path = Path('../data/processed/nasdaq_website_lite.csv')

# List of tickers to keep for the lightweight version
target_tickers = ['AAPL', 'TSLA', 'QCOM', 'NFLX', 'AMZN']

# Lite dataset for website
# Filter only selected tickers
filtered_df = cleaned_df[cleaned_df['ticker'].isin(target_tickers)].copy()

print("Saving filtered data for the website...")
filtered_df.to_csv(lite_output_path, index=False)
lite_size_mb = lite_output_path.stat().st_size / 1e6
print(f"Saved {len(filtered_df):,} rows to {lite_output_path}")
print(f"File size: {lite_size_mb:.4f} MB")

Overwriting existing file (1.38 GB)
Saving cleaned data...
Saved 12,993,961 rows to ../data/processed/nasdaq_processed.csv
File size: 1.38 GB

Columns saved: Date, Open, High, Low, Close, Adj Close, Volume, ticker

Cleaned data saved to ../data/processed/nasdaq_processed.csv
Filtering for: AAPL, TSLA, QCOM, NFLX, AMZN...
Saving filtered data for the website...
Saved 29,748 rows to ../data/processed/nasdaq_website_lite.csv
File size: 3.2836 MB
